In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import json
import time

start_time = time.time()
print("XGBoost + LightGBM Ensemble - CURATED HOTSPOT FEATURES")
print("="*80)

df = pd.read_csv('MSKMET_Hotspots.csv', index_col=0)

print(f"Loaded: {df.shape[0]} samples, {df.shape[1]} features")

df = df[df['Primary_Site_Target'].notna()].copy()

non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
if 'Primary_Site_Target' in non_numeric_cols:
    non_numeric_cols.remove('Primary_Site_Target')

for col in non_numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

numeric_cols = ['AGE_AT_SEQUENCING', 'TMB_NONSYNONYMOUS', 'MSI_SCORE']
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

X = df.drop(columns=['Primary_Site_Target'])
y = df['Primary_Site_Target']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

print(f"Classes: {num_classes}")

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y_encoded, test_size=0.20, stratify=y_encoded, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, stratify=y_train_full, random_state=42
)

nonzero = X_train.sum(axis=0) != 0
keep_cols = nonzero[nonzero].index

X_train = X_train[keep_cols]
X_val = X_val[keep_cols]
X_test = X_test[keep_cols]

print(f"\nAfter filtering: {len(keep_cols)} features")
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

feature_types = {
    'LOF': len([c for c in keep_cols if c.endswith('_LOF')]),
    'Non-LOF': len([c for c in keep_cols if c.endswith('_NON_LOF')]),
    'Amplification': len([c for c in keep_cols if c.endswith('_AMP')]),
    'Deletion': len([c for c in keep_cols if c.endswith('_DEL')]),
    'Fusion': len([c for c in keep_cols if c.endswith('_FUSION')]),
    'Gene Hotspot': len([c for c in keep_cols if c.endswith('_HOTSPOT') and 'AA' not in c and 'SPLICE' not in c]),
    'AA Hotspot': len([c for c in keep_cols if c.endswith('_AA_HOTSPOT')]),
    'Splice Hotspot': len([c for c in keep_cols if c.endswith('_SPLICE_HOTSPOT')])
}

print("\nFeature Type Breakdown:")
for ft, count in feature_types.items():
    print(f"  {ft:<20}: {count}")

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
sample_weights = np.array([class_weights[i] for i in y_train])

xgb_params = {
    'objective': 'multi:softprob',
    'num_class': num_classes,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cuda',
    'max_depth': 10,
    'learning_rate': 0.05,
    'n_estimators': 500,
    'subsample': 0.8,
    'colsample_bytree': 0.75,
    'min_child_weight': 2,
    'gamma': 0.3,
    'reg_lambda': 2.0,
    'reg_alpha': 0.15,
    'random_state': 42
}

print("\nTraining XGBoost...")
xgb_model = xgb.XGBClassifier(**xgb_params)
xgb_model.fit(X_train.values, y_train, sample_weight=sample_weights, verbose=False)

lgb_params = {
    'objective': 'multiclass',
    'num_class': num_classes,
    'max_depth': 10,
    'learning_rate': 0.05,
    'n_estimators': 500,
    'subsample': 0.8,
    'colsample_bytree': 0.75,
    'min_child_weight': 2,
    'reg_lambda': 2.0,
    'reg_alpha': 0.15,
    'random_state': 42,
    'verbose': -1,
    'class_weight': 'balanced'
}

print("Training LightGBM...")
lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(X_train.values, y_train)

print("\nEvaluating models...")

models = {'XGBoost': xgb_model, 'LightGBM': lgb_model}
results = {}

for name, model in models.items():
    y_pred = model.predict(X_test.values)
    y_pred_proba = model.predict_proba(X_test.values)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    top3 = top_k_accuracy_score(y_test, y_pred_proba, k=3)

    results[name] = {'accuracy': acc, 'f1_macro': f1, 'top3': top3, 'predictions': y_pred_proba}
    print(f"{name}: Accuracy={acc:.4f} ({acc*100:.2f}%), F1={f1:.4f}, Top-3={top3:.4f}")

weights = np.array([results[name]['accuracy'] for name in models.keys()])
weights = weights / weights.sum()

print(f"\nEnsemble weights: XGBoost={weights[0]:.3f}, LightGBM={weights[1]:.3f}")

ensemble_proba = weights[0] * results['XGBoost']['predictions'] + weights[1] * results['LightGBM']['predictions']
ensemble_pred = ensemble_proba.argmax(axis=1)

ensemble_acc = accuracy_score(y_test, ensemble_pred)
ensemble_f1 = f1_score(y_test, ensemble_pred, average='macro')
ensemble_top3 = top_k_accuracy_score(y_test, ensemble_proba, k=3)
ensemble_top5 = top_k_accuracy_score(y_test, ensemble_proba, k=5)

xgb_importance = pd.DataFrame({
    'feature': keep_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 Most Important Features:")
for idx, row in xgb_importance.head(15).iterrows():
    print(f"  {row['feature']:<50} {row['importance']:.4f}")

elapsed = time.time() - start_time

print("\n" + "="*80)
print("FINAL ENSEMBLE RESULTS (CURATED FEATURES)")
print("="*80)
print(f"Ensemble Accuracy:  {ensemble_acc:.4f} ({ensemble_acc*100:.2f}%)")
print(f"Top-3 Accuracy:     {ensemble_top3:.4f} ({ensemble_top3*100:.2f}%)")
print(f"Top-5 Accuracy:     {ensemble_top5:.4f} ({ensemble_top5*100:.2f}%)")
print(f"F1-Macro:           {ensemble_f1:.4f}")
print(f"\nTraining Time:      {elapsed/60:.1f} minutes")

final_results = {
    'ensemble': {
        'test_accuracy': float(ensemble_acc),
        'top3_accuracy': float(ensemble_top3),
        'top5_accuracy': float(ensemble_top5),
        'f1_macro': float(ensemble_f1)
    },
    'individual_models': {
        name: {'accuracy': float(r['accuracy']), 'f1': float(r['f1_macro'])}
        for name, r in results.items()
    },
    'feature_breakdown': feature_types,
    'runtime_minutes': elapsed/60
}

with open('curated_ensemble_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

xgb_importance.to_csv('feature_importance_curated.csv', index=False)

print("\nResults saved!")

XGBoost + LightGBM Ensemble - CURATED HOTSPOT FEATURES
Loaded: 25775 samples, 8014 features
Classes: 27

After filtering: 6899 features
Train: 17527, Val: 3093, Test: 5155

Feature Type Breakdown:
  LOF                 : 960
  Non-LOF             : 479
  Amplification       : 474
  Deletion            : 467
  Fusion              : 2369
  Gene Hotspot        : 196
  AA Hotspot          : 1863
  Splice Hotspot      : 78

Training XGBoost...
Training LightGBM...

Evaluating models...


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [19:46:08] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


XGBoost: Accuracy=0.7263 (72.63%), F1=0.5529, Top-3=0.8780


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM: Accuracy=0.7193 (71.93%), F1=0.5444, Top-3=0.8694

Ensemble weights: XGBoost=0.502, LightGBM=0.498

Top 15 Most Important Features:
  VHL_MUT                                            0.0097
  IDH1.R132C_AA_HOTSPOT                              0.0097
  TMPRSS2_FUSION                                     0.0081
  KIT_MUT                                            0.0057
  VHL_LOF                                            0.0052
  VTCN1_MUT                                          0.0052
  SPOP_HOTSPOT                                       0.0042
  CCNE1_DEL                                          0.0040
  FGFR3.S249C_AA_HOTSPOT                             0.0038
  HIST1H3E_NON_LOF                                   0.0038
  RRAS_DEL                                           0.0037
  CTNNB1.S45F_AA_HOTSPOT                             0.0036
  HRAS.Q61L_AA_HOTSPOT                               0.0036
  GATA3_LOF                                          0.0036
  FGF19_MUT       